# NB01 - CMS ASP NDC-HCPCS Crosswalk
### Cross-Benefit Duplicate Drug Billing Detection - Foundation Layer

**Schema:** `cross_benefit_audit`
**Table:** `cms_ndc_hcpcs_crosswalk` (full), `target_hcpcs_crosswalk` (project scope)
**Source:** CMS October 2026 ASP NDC-HCPCS Crosswalk (effective 2026-10-01 to 2026-12-31, published 2026-09-17)

---

## Why a crosswalk is the foundation

Detecting the same drug billed twice - once through the pharmacy benefit and once through the medical benefit - is architecturally impossible without solving a drug identity problem first.

Pharmacy claims travel on **NCPDP** transactions and identify drugs with **NDC codes** (11-digit: labeler + product + package). Medical claims travel on CMS-1500 or UB-04 forms and identify drugs with **HCPCS J/Q codes** (e.g., J9271 = pembrolizumab 1 mg).

**The two systems share no common drug identifier by design**.

CMS publishes a quarterly **Average Sales Price (ASP) NDC-HCPCS Crosswalk** that links every pharmacy-dispensable NDC to the HCPCS code a provider would bill when administering that same drug. It also provides **unit conversion factors** (`BILLUNITS`, `BILLUNITSPKG`) so that dose quantities stated in different units can be reconciled.

- https://www.cms.gov/medicare/payment/part-b-drugs/asp-pricing-files<br>
- https://www.cms.gov/files/zip/october-2026-ndc-hcpcs-crosswalk-final.zip<br>

### This notebook:
1. Loads and explores the raw October 2026 ASP crosswalk CSV
2. Runs systematic data quality checks across all 7,403 rows
3. Characterises anomalies and documents what is a real error vs. intentional CMS methodology
4. Applies a cleaning pipeline
5. Writes the result to a **Delta table** in the `cross_benefit_audit` schema - both the full file and a filtered table scoped to the 6 specialty drugs this project targets

## Project drug scope

| Therapeutic area | Drug (molecule) | Originator J/Q-code | Biosimilar / alt-formulation codes |
|---|---|---|---|
| Ophthalmology | Ranibizumab | J2778 | Q5124 (Byooviz), Q5128 (Cimerli) |
| Ophthalmology | Aflibercept | J0178 | Q5138 (Wezlana) |
| Oncology | Bevacizumab | J9035 | Q5107 (Mvasi), Q5118 (Zirabev) |
| Oncology | Pembrolizumab | J9271 | J9277 (Keytruda Qlex SC) |
| Autoimmune | Infliximab | J1745¹ | Q5103 (Inflectra), Q5104 (Renflexis), Q5121 (Avsola) |
| Neurology | Ocrelizumab | J2350 | J2351 (Ocrevus Zunovo SC) |

¹ J1745 short descriptor reads *"Infliximab not biosimil 10mg"* - biosimilars are intentionally coded under separate Q-codes.

## 0 - Setup

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType, IntegerType, DateType
)
import pandas as pd
import re
from datetime import date

spark = SparkSession.builder.getOrCreate()

RAW_CSV = "/Volumes/workspace/cross_benefit_audit/raw_files/october_2026_asp_ndc_hcpcs_crosswalk.csv"

SOURCE_EFFECTIVE_START = date(2026, 10, 1)
SOURCE_EFFECTIVE_END   = date(2026, 12, 31)
SOURCE_PUBLISHED       = date(2026, 9, 17)

print("Setup complete.")
print(f"Source file  : {RAW_CSV}")
print(f"Effective    : {SOURCE_EFFECTIVE_START} → {SOURCE_EFFECTIVE_END}")
print(f"Published    : {SOURCE_PUBLISHED}")

Setup complete.
Source file  : /Volumes/workspace/cross_benefit_audit/raw_files/october_2026_asp_ndc_hcpcs_crosswalk.csv
Effective    : 2026-10-01 → 2026-12-31
Published    : 2026-09-17


---
## 1 - Raw file exploration

Before loading with `skiprows`, let's inspect the preamble so the encoding choice and header offset are documented in-notebook.

In [0]:
# Read the first 12 raw lines so we can see the preamble (first 8)
with open(RAW_CSV, encoding="latin-1") as fh:
    raw_head = [fh.readline().rstrip("\n") for _ in range(12)]

print("RAW PREAMBLE (first 12 lines):")
print("=" * 80)
for i, line in enumerate(raw_head):
    print(f"  row {i:02d}: {line[:120]}")

RAW PREAMBLE (first 12 lines):
  row 00: October 2026 ASP NDC - HCPCS Crosswalk for Medicare Part B Drugs,,,,,,,,,
  row 01: "     Effective October 1, 2026 through December 31, 2026",,,,,,,,,
  row 02: ,,,,,,,,,
  row 03: The ASP crosswalks are maintained by the Division of Data Analysis and Market Based Pricing to support ASP-based Medicar
  row 04: ,,,,,,,,,
  row 05: ,,,,,,,,,
  row 06: ,,,,,,,,,
  row 07: ,,,,,,,,,
  row 08: _2026_CODE,Short Description,LABELER NAME,NDC,Drug Name,HCPCS dosage,PKG SIZE,PKG QTY,BILLUNITS,BILLUNITSPKG
  row 09: 90371,Hep b ig im,"Grifols USA, LLC",13533-0636-01,HyperHepB ,1 ML,1,1,1,1
  row 10: 90371,Hep b ig im,"Grifols USA, LLC",13533-0636-03,HyperHepB ,1 ML,0.5,1,0.5,1
  row 11: 90371,Hep b ig im,"Grifols USA, LLC",13533-0636-05,HyperHepB,1 ML,5,1,5,5


**Observations:**
- Rows 0-7: CMS title, effective period, blank separators, description text
- Row 8: column headers (`_2026_CODE`, `Short Description`, ...)
- Row 9+: data

This is Windows-1252 encoding (`latin-1` compatible) - confirmed by the presence of non-breaking space `\xa0` characters in the preamble. UTF-8 would raise a `UnicodeDecodeError` at byte position 0xa0.

Load parameters: `skiprows=8`, `encoding="latin-1"`.

In [0]:
# Load with correct parameters
df_raw = pd.read_csv(
    RAW_CSV,
    skiprows=8,
    dtype=str,            # read everything as string first
    encoding="latin-1",
)
df_raw.rename(columns={
    '_2026_CODE':       'HCPCS_CODE',
    'Short Description':'Short_Description',
    'LABELER NAME':     'LABELER_NAME',
    'Drug Name':        'Drug_Name',
    'HCPCS dosage':     'HCPCS_dosage',
    'PKG SIZE':         'PKG_SIZE',
    'PKG QTY':          'PKG_QTY',
}, inplace=True)

print(f"Rows loaded  : {len(df_raw):,}")
print(f"Columns      : {list(df_raw.columns)}")

Rows loaded  : 7,403
Columns      : ['HCPCS_CODE', 'Short_Description', 'LABELER_NAME', 'NDC', 'Drug_Name', 'HCPCS_dosage', 'PKG_SIZE', 'PKG_QTY', 'BILLUNITS', 'BILLUNITSPKG']


In [0]:
# Preview
df_raw.head(5)

,HCPCS_CODE,Short_Description,LABELER_NAME,NDC,Drug_Name,HCPCS_dosage,PKG_SIZE,PKG_QTY,BILLUNITS,BILLUNITSPKG
0,90371,Hep b ig im,"Grifols USA, LLC",13533-0636-01,HyperHepB,1 ML,1,1,1,1
1,90371,Hep b ig im,"Grifols USA, LLC",13533-0636-03,HyperHepB,1 ML,0.5,1,0.5,1
2,90371,Hep b ig im,"Grifols USA, LLC",13533-0636-05,HyperHepB,1 ML,5,1,5,5
3,90371,Hep b ig im,"ADMA Biologics, Inc",69800-4202-01,Nabi-Hb,1 ML,1,1,1,1
4,90371,Hep b ig im,"ADMA Biologics, Inc",69800-4203-01,Nabi-Hb,1 ML,5,1,5,5


---
## 2 - Data quality checks

We run checks in a logical order, documenting each finding before moving to cleaning. Data Quality (DQ) findings enumerated as DQ-01, DQ-02 etc.

### 2.1 - Schema and null coverage

In [0]:
null_counts = df_raw.isnull().sum()
total = len(df_raw)
print(f"{'Column':<22} {'Nulls':>6} {'% null':>8}")
print("-" * 40)
for col, n in null_counts.items():
    print(f"{col:<22} {n:>6} {100*n/total:>7.2f}%")

Column                  Nulls   % null
----------------------------------------
HCPCS_CODE                  0    0.00%
Short_Description           0    0.00%
LABELER_NAME                0    0.00%
NDC                         0    0.00%
Drug_Name                   0    0.00%
HCPCS_dosage                0    0.00%
PKG_SIZE                    0    0.00%
PKG_QTY                     0    0.00%
BILLUNITS                   0    0.00%
BILLUNITSPKG                0    0.00%


### 2.2 - NDC format validation

The standard CMS dash-formatted NDC is `LLLLL-PPPP-PP` (5-4-2, 13 chars including dashes). Any row that doesn't match this pattern is either a scan artifact, a legacy barcode, or an alternate manufacturer ID.

In [0]:
NDC_PATTERN = re.compile(r"^\d{5}-\d{4}-\d{2}$") # LLLLL-PPPP-PP pattern

df_raw["ndc_valid"] = df_raw["NDC"].str.match(NDC_PATTERN, na=False)
ndc_bad = df_raw[~df_raw["ndc_valid"]]# Catches the ones that do not match the pattern

print(f"Valid NDC format   : {df_raw['ndc_valid'].sum():>5} / {len(df_raw):>5}")
print(f"Invalid NDC format : {(~df_raw['ndc_valid']).sum():>5}")

if len(ndc_bad):
    print()
    print("Invalid rows:")
    print(ndc_bad[["HCPCS_CODE", "Short_Description", "LABELER_NAME", "NDC", "Drug_Name"]].to_string(index=False))

Valid NDC format   :  7402 /  7403
Invalid NDC format :     1

Invalid rows:
HCPCS_CODE     Short_Description  LABELER_NAME          NDC Drug_Name
     J7331 Synojoynt, inj., 1 mg Arthrex, Inc. 888867413689 SynoJoynt


**Finding DQ-01 - Non-standard NDC (1 row)**

| Field | Value |
|---|---|
| HCPCS | J7331 (Synojoynt) |
| Labeler | Arthrex, Inc. |
| NDC | `888867413689` (12 digits instead of 13, no dashes) |

`888867413689` is a GS1-128 / HRI barcode format used by some specialty distributors, not a CMS-formatted NDC. It cannot be matched against pharmacy claims.

**Action:**<br>
Exclude from the clean table; log as `EXCLUDED_NDC_FORMAT`.
J7331 Synojoynt (hyaluronic acid) is outside our project scope regardless.

### 2.3 - Numeric column parsing

In [0]:
NUM_COLS = ["PKG_SIZE", "PKG_QTY", "BILLUNITS", "BILLUNITSPKG"]

df_num = df_raw.copy()
for col in NUM_COLS:
    df_num[col + "_num"] = pd.to_numeric(df_num[col], errors="coerce")

parse_errors = {
    col: df_num[col + "_num"].isnull().sum() - df_num[col].isnull().sum()
    for col in NUM_COLS
}

print("Numeric parse errors (coerced to NaN):")
for col, n in parse_errors.items():
    print(f"  {col:<16}: {n}")

Numeric parse errors (coerced to NaN):
  PKG_SIZE        : 0
  PKG_QTY         : 0
  BILLUNITS       : 0
  BILLUNITSPKG    : 0


**Finding DQ-02 - Numeric columns parse cleanly**

Zero parse errors. All four numeric fields (`PKG SIZE`, `PKG QTY`,
`BILLUNITS`, `BILLUNITSPKG`) can be cast to `DOUBLE` without data loss.

### 2.4 - Duplicate NDC rows - true duplicates vs. multi-HCPCS mapping

In [0]:
# How many NDCs appear more than once?
ndc_counts = df_raw.groupby("NDC").size().reset_index(name="n_hcpcs")
multi_ndc  = ndc_counts[ndc_counts["n_hcpcs"] > 1]

dup_all_rows = df_raw[df_raw["NDC"].isin(multi_ndc["NDC"])]

# Are any of these the SAME NDC+HCPCS pair (true duplicates)?
true_dups = df_raw[df_raw.duplicated(subset=["NDC", "HCPCS_CODE"], keep=False)]

print(f"NDCs appearing more than once : {len(multi_ndc):>5}")
print(f"Rows involved                 : {len(dup_all_rows):>5}")
print()
print(f"True duplicates (same NDC + same HCPCS) : {len(true_dups)}")
print(f"  - All multi-NDC rows are legitimate multi-HCPCS mappings.")
print()
print("Top 10 most-mapped NDCs:")
print(
    dup_all_rows.groupby("NDC")
    .agg(n=("HCPCS_CODE", "nunique"), codes=("HCPCS_CODE", lambda x: ", ".join(sorted(x.unique()))))
    .sort_values("n", ascending=False)
    .head(10)
    .to_string()
)

NDCs appearing more than once :   138
Rows involved                 :   319

True duplicates (same NDC + same HCPCS) : 0
  - All multi-NDC rows are legitimate multi-HCPCS mappings.

Top 10 most-mapped NDCs:
               n                codes
NDC                                  
00338-9810-24  3  J7030, J7040, J7050
65219-0224-50  3  J7030, J7040, J7050
00338-0049-03  3  J7030, J7040, J7050
00338-0049-04  3  J7030, J7040, J7050
65219-0222-25  3  J7030, J7040, J7050
00990-7983-02  3  J7030, J7040, J7050
00338-9543-01  3  J7030, J7040, J7050
00338-9543-02  3  J7030, J7040, J7050
00338-9543-03  3  J7030, J7040, J7050
00338-9543-04  3  J7030, J7040, J7050


In [0]:
ndc_counts[ndc_counts["n_hcpcs"] > 1]

,NDC,n_hcpcs
287,00069-1305-10,2
288,00069-1306-10,2
289,00069-1307-10,2
290,00069-1308-10,2
291,00069-1309-04,2
...,...,...
5241,67457-0212-02,2
6072,70512-0841-26,3
6073,70512-0841-51,3
6074,70512-0841-61,3


In [0]:
true_dups


,HCPCS_CODE,Short_Description,LABELER_NAME,NDC,Drug_Name,HCPCS_dosage,PKG_SIZE,PKG_QTY,BILLUNITS,BILLUNITSPKG,ndc_valid


**Finding DQ-03 - 319 multi-HCPCS NDC rows and zero true duplicates**

319 rows involve NDCs that appear under more than one HCPCS code.<br>
**All 319 are legitimate clinical multi-mappings**, meaning that the same NDC is billable under different codes depending on clinical indication.

Example: **Retacrit** (epoetin alfa biosimilar)
- Q5105 - ESRD (end-stage renal disease) use
- Q5106  non-ESRD use (oncology chemotherapy-induced anemia)

It involves the same product, the same package but there is a different billing code depending on the clinical indication documented on the claim.

**Detection implication:**<br>
When querying the crosswalk during duplicate detection, a pharmacy NDC may resolve to multiple HCPCS codes. The detection logic must consider all mapped codes, not just the first hit!

**Action:**<br>
Retain all rows. Do not deduplicate on NDC alone.

### 2.5 - BILLUNITS × PKG_QTY ≠ BILLUNITSPKG

Per the CMS data dictionary:
- **BILLUNITS** = billable units per package item (per vial, syringe, or tablet).
- **BILLUNITSPKG** = billable units per NDC. In theory, the full shelf pack = BILLUNITS × PKG_QTY.

First we test the arithmetic, then classify every mismatch.

In [0]:
import math

for col in NUM_COLS:
    df_num[col + "_num"] = pd.to_numeric(df_num[col], errors="coerce")

df_num["bu_expected"] = df_num["BILLUNITS_num"] * df_num["PKG_QTY_num"]
df_num["bu_diff"] = (df_num["BILLUNITSPKG_num"] - df_num["bu_expected"]).abs()

mismatch = df_num[df_num["bu_diff"] > 0.001].copy()

print(f"Rows where BILLUNITSPKG ≠ BILLUNITS × PKG_QTY : {len(mismatch)} / {len(df_num)}")

# Classify into three buckets
def classify_mismatch(row):
    actual   = row["BILLUNITSPKG_num"]
    expected = row["bu_expected"]
    diff     = actual - expected
    if diff < 0:
        return "UNDER_CMS_DIRECT"       # CMS set BILLUNITSPKG = BILLUNITS (PKG_QTY ignored)
    ceil_exp = math.ceil(expected)
    if abs(actual - ceil_exp) <= 0.5:
        return "OVER_CEILING_ROUND"     # fractional BILLUNITS rounded up to nearest integer
    return "OVER_SHELF_PACK_TOTAL"      # BILLUNITSPKG equals the full physical content of shelf pack

mismatch["dq_class"] = mismatch.apply(classify_mismatch, axis=1)
print()
print(mismatch["dq_class"].value_counts().to_string())

Rows where BILLUNITSPKG ≠ BILLUNITS × PKG_QTY : 208 / 7403

dq_class
OVER_CEILING_ROUND       139
OVER_SHELF_PACK_TOTAL     49
UNDER_CMS_DIRECT          20


**Finding DQ-04 - 208 arithmetic mismatches - all three categories are intentional CMS methodology**

| Category | Count | Explanation |
|---|---|---|
| `UNDER_CMS_DIRECT` | 20 | Oral drugs where CMS set BILLUNITSPKG = BILLUNITS, effectively treating each tablet/capsule as its own NDC unit. Most are bottle-packed oral immunosuppressants (mycophenolate, macrolimus, prednisone, Emend). |
| `OVER_CEILING_ROUND` | 139 | Fractional BILLUNITS (e.g. 0.5 units, or 222.33 units) are ceiling-rounded so that BILLUNITSPKG is always a whole integer. CMS intentional. |
| `OVER_SHELF_PACK_TOTAL` | 49 | BILLUNITSPKG reflects the total physical content of the NDC package. Examples: etoposide vials (10 mL × 10 mg/mL → BILLUNITSPKG = 100), vancomycin premixed IV bags (total mg in the full bag), oral repackaged unit-dose packs (ondansetron tablet counts varying per strip size). |

**None of these are data errors!**<br>

CMS sets `BILLUNITSPKG` directly from the physical NDC and clinical billing convention, independent of the formula.

**I'm using `BILLUNITSPKG` as the authoritative billing-unit.**
Here, I did not compute it from `BILLUNITS × PKG_QTY`. The rows in which `BILLUNITS × PKG_QTY ≠ BILLUNITSPKG` were flagged with the classification label.

Zero of the 31 target rows (selected drugs for this project) appear in the mismatch set.

In [0]:
# Are any target-drug rows in the mismatch set?
TARGET_CODES = [
    "J0178", "J1745", "J2350", "J2351",
    "J2778", "J9035", "J9271", "J9277",
    "Q5103", "Q5104", "Q5107", "Q5118",
    "Q5121", "Q5124", "Q5128", "Q5138",
]
target_in_mismatch = mismatch[mismatch["HCPCS_CODE"].isin(TARGET_CODES)]
print(f"Target drug rows in mismatch set: {len(target_in_mismatch)}")  # should be 0

Target drug rows in mismatch set: 0


### 2.6 - Ranibizumab / biosimilar billing-unit split

An anomaly surfaced during the target drug review:<br>
**Lucentis PFS shows BILLUNITS = 3 while the standard Lucentis vial shows BILLUNITS = 5**, despite both having the same HCPCS dosage descriptor ("0.1 MG") and the same PKG SIZE (0.05 mL).

In [0]:
rani_group = df_num[df_num["HCPCS_CODE"].isin(["J2778", "Q5124", "Q5128"])].copy()
print("All ranibizumab products (originator + biosimilars):")
print(
    rani_group[[
        "HCPCS_CODE", "Drug_Name", "NDC",
        "PKG_SIZE_num", "BILLUNITS_num", "BILLUNITSPKG_num"
    ]].sort_values(["HCPCS_CODE", "NDC"]).to_string(index=False)
)

All ranibizumab products (originator + biosimilars):
HCPCS_CODE    Drug_Name           NDC  PKG_SIZE_num  BILLUNITS_num  BILLUNITSPKG_num
     J2778     Lucentis 50242-0080-03          0.05            5.0               5.0
     J2778 Lucentis PFS 50242-0082-03          0.05            3.0               3.0
     Q5124      Byooviz 64406-0019-01          0.05            5.0               5.0
     Q5124      Byooviz 82667-0018-05          0.05            5.0               5.0
     Q5128      Cimerli 61314-0624-94          0.05            3.0               3.0
     Q5128      Cimerli 61314-0625-94          0.05            5.0               5.0
     Q5128      Cimerli 70114-0440-01          0.05            3.0               3.0
     Q5128      Cimerli 70114-0441-01          0.05            5.0               5.0


**Finding DQ-05 - Two-concentration split is clinically correct**

J2778, the J code, is billed per **0.1 mg** of ranibizumab.

| Product | Concentration | Vial content (0.05 mL) | Billable units |
|---|---|---|---|
| Lucentis standard | 10 mg/mL | 0.05 mL × 10 = 0.5 mg | 0.5 ÷ 0.1 = **5** |
| Lucentis PFS | 6 mg/mL | 0.05 mL × 6 = 0.3 mg | 0.3 ÷ 0.1 = **3** |

Cimerli (Q5128) ships in both concentrations (two NDC pairs: `-94` = 6 mg/mL, `-01` = 10 mg/mL), explaining its internal BILLUNITS variation (3 vs. 5).<br>
Byooviz (Q5124) is only the 10 mg/mL formulation (BILLUNITS = 5).

**Detection implication:**<br>
If the pharmacy dispenses Lucentis PFS (3 units) and the medical claim is billed as J2778 × 5 units, the dose quantities won't match on a strict arithmetic comparison. The crosswalk NDC is the ground-truth resolver - always look up the specific NDC, not just the parent HCPCS code.

**Action:**<br>
Retain both rows and add a `concentration_mg_per_ml` derived column.

### 2.7 - Biosimilar HCPCS code separation

A critical structural feature of the CMS crosswalk is that **biosimilars are NOT coded under the originator's J-code.**
They receive separate Q-codes. This is not a data quality issue, it is CMS policy that has major consequences for naive duplicate detection.

In [0]:
# The infliximab case
infl_rows = df_raw[
    df_raw["Drug_Name"].str.contains(
        "nfliximab|inflectra|renflexis|avsola", na=False, case=False #originator and biosimilars
    )
]
print(
    infl_rows[["HCPCS_CODE", "Short_Description", "Drug_Name", "LABELER_NAME", "NDC"]]
    .sort_values("HCPCS_CODE")
    .to_string(index=False)
)

HCPCS_CODE            Short_Description  Drug_Name          LABELER_NAME           NDC
     J1745 Infliximab not biosimil 10mg Infliximab Janssen Biotech, Inc. 57894-0160-01
     Q5103         Injection, inflectra  Inflectra            Pfizer Inc 00069-0809-01
     Q5104         Injection, renflexis  Renflexis           Organon LLC 78206-0162-01
     Q5121           Inj. avsola, 10 mg     Avsola             Amgen Inc 55513-0670-01
     Q5121           Inj. avsola, 10 mg     Avsola             Amgen Inc 55513-0670-21


**Finding DQ-06 - Biosimilars coded under Q-codes, not parent J-codes**

| Drug | Originator code | Biosimilar codes |
|---|---|---|
| Infliximab | J1745 ("not biosimil") | Q5103 Inflectra, Q5104 Renflexis, Q5121 Avsola |
| Bevacizumab | J9035 | Q5107 Mvasi, Q5118 Zirabev |
| Ranibizumab | J2778 | Q5124 Byooviz, Q5128 Cimerli |
| Aflibercept | J0178 | Q5138 Wezlana |
| Pembrolizumab | J9271 | *(no biosimilars approved as of Q4 2026)* |
| Ocrelizumab | J2350 | *(no biosimilars approved as of Q4 2026)* |

**Detection scenario:**<br>
A member on a specialty pharmacy benefit fills Mvasi (bevacizumab biosimilar) at the pharmacy. The pharmacy claim carries the Mvasi NDC, which maps to
**Q5107** in this crosswalk. The infusion center bills the medical claim as **J9035** (the originator Avastin code). A detection engine matching only
J9035-to-J9035 would miss this entirely.

This project's target HCPCS set explicitly includes all Q-codes so that originator-vs-biosimilar cross-billing is detectable.

**New delivery formulations** (pharmacologically equivalent but with a different device or formulation):<br>
- J2351: Ocrevus Zunovo (ocrelizumab SC - same molecule as J2350 IV)
- J9277: Keytruda Qlex (pembrolizumab + hyaluronidase SC - same molecule as J9271 IV)

These are included in our scope because a member could fill the subcutaneous formulation at pharmacy and have the IV equivalent billed on the medical side.

---
## 3 - DQ summary

In [0]:
import json

dq_findings = [
    {
        "id": "DQ-01",
        "check": "NDC format validation",
        "rows_affected": 1,
        "severity": "LOW",
        "description": "SynoJoynt (J7331, Arthrex) carries NDC 888867413689 - a 12-digit GS1-128 barcode, not a CMS 5-4-2 formatted NDC.",
        "action": "EXCLUDE from clean table; log as EXCLUDED_NDC_FORMAT.",
        "in_scope": False,
    },
    {
        "id": "DQ-02",
        "check": "Numeric column parseability",
        "rows_affected": 0,
        "severity": "PASS",
        "description": "All four numeric fields (PKG SIZE, PKG QTY, BILLUNITS, BILLUNITSPKG) parse cleanly to DOUBLE. No coercion errors.",
        "action": "None.",
        "in_scope": True,
    },
    {
        "id": "DQ-03",
        "check": "Duplicate NDC rows",
        "rows_affected": 319,
        "severity": "INFO",
        "description": "319 rows involve NDCs that appear under >1 HCPCS code. Zero true duplicates (same NDC + same HCPCS). All are legitimate indication-specific multi-mappings (e.g. Retacrit: Q5105 for ESRD, Q5106 for non-ESRD).",
        "action": "Retain all rows. Detection logic must consider all mapped HCPCS codes for a given NDC.",
        "in_scope": True,
    },
    {
        "id": "DQ-04",
        "check": "BILLUNITS × PKG_QTY vs BILLUNITSPKG arithmetic",
        "rows_affected": 208,
        "severity": "INFO",
        "description": (
            "208 rows where BILLUNITSPKG ≠ BILLUNITS × PKG_QTY: "
            "20 UNDER (CMS set BILLUNITSPKG = BILLUNITS directly, for oral unit-dose drugs); "
            "139 OVER-ceiling-round (fractional BILLUNITS rounded up to integer); "
            "49 OVER-shelf-pack (BILLUNITSPKG = total physical content of NDC package, e.g. Etoposide multi-dose vials, premixed IV bags, oral repackaged strips). "
            "None of our 6 target drugs are affected."
        ),
        "action": "Use BILLUNITSPKG as-supplied. Do not recompute. Tag mismatch rows for transparency.",
        "in_scope": False,  # target drugs unaffected
    },
    {
        "id": "DQ-05",
        "check": "Ranibizumab / biosimilar billing-unit split",
        "rows_affected": 6,
        "severity": "INFO",
        "description": (
            "Lucentis PFS (BILLUNITS=3) vs standard Lucentis (BILLUNITS=5) for same 0.1 MG HCPCS dosage. "
            "Explained by different concentrations: PFS is 6 mg/mL (→ 0.3 mg in 0.05 mL = 3 units); "
            "standard is 10 mg/mL (→ 0.5 mg = 5 units). "
            "Cimerli (Q5128) ships in both concentrations; both NDC variants present."
        ),
        "action": "Retain all rows. Detection must use NDC-level BILLUNITS, not HCPCS-level aggregate.",
        "in_scope": True,
    },
    {
        "id": "DQ-06",
        "check": "Biosimilar HCPCS code separation",
        "rows_affected": 0,
        "severity": "INFO",
        "description": (
            "CMS codes biosimilars under Q-codes, not the originator J-code. "
            "J1745 is explicitly 'not biosimil'. "
            "This is not a data error but has critical detection implications."
        ),
        "action": (
            "Expand target HCPCS set to include Q5103/Q5104/Q5121 (infliximab), "
            "Q5107/Q5118 (bevacizumab), Q5124/Q5128 (ranibizumab), Q5138 (aflibercept), "
            "J2351 (ocrelizumab SC), J9277 (pembrolizumab SC)."
        ),
        "in_scope": True,
    },
]

print("=" * 80)
print(f"{'ID':<8} {'Rows':>6}  {'Severity':<12}  Check")
print("=" * 80)
for f in dq_findings:
    print(f"  {f['id']:<8} {f['rows_affected']:>6}  {f['severity']:<12}  {f['check']}")
print()
print("Files in scope for clean Delta table: 7,402 rows (1 excluded, DQ-01)")

ID         Rows  Severity      Check
  DQ-01         1  LOW           NDC format validation
  DQ-02         0  PASS          Numeric column parseability
  DQ-03       319  INFO          Duplicate NDC rows
  DQ-04       208  INFO          BILLUNITS × PKG_QTY vs BILLUNITSPKG arithmetic
  DQ-05         6  INFO          Ranibizumab / biosimilar billing-unit split
  DQ-06         0  INFO          Biosimilar HCPCS code separation

Files in scope for clean Delta table: 7,402 rows (1 excluded, DQ-01)


---
## 4 - Cleaning pipeline

In [0]:
# Step 1- Cast numeric columns
df_clean = df_raw.copy()
for col in NUM_COLS:
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

# Step 2 - Exclude the malformed NDC row
ndc_excluded = df_clean[~df_clean["NDC"].str.match(NDC_PATTERN, na=False)].copy()
ndc_excluded["exclusion_reason"] = "EXCLUDED_NDC_FORMAT"
df_clean = df_clean[df_clean["NDC"].str.match(NDC_PATTERN, na=False)].copy()
print(f"After NDC format exclusion: {len(df_clean):,} rows  (excluded: {len(ndc_excluded)})")

# Step 3 - Strip leading/trailing whitespace from string columns
str_cols = ["HCPCS_CODE", "Short_Description", "LABELER_NAME", "NDC",
            "Drug_Name", "HCPCS_dosage"]

for col in str_cols:
    df_clean[col] = df_clean[col].str.strip()

# Step 4 - Tag arithmetic mismatch category
def tag_bu_mismatch(row):
    expected = row["BILLUNITS"] * row["PKG_QTY"]
    diff = abs(row["BILLUNITSPKG"] - expected)
    if diff <= 0.001:
        return "OK"
    elif row["BILLUNITSPKG"] < expected:
        return "UNDER_CMS_DIRECT"
    else:
        ceil_exp = math.ceil(expected)
        if abs(row["BILLUNITSPKG"] - ceil_exp) <= 0.5:
            return "OVER_CEILING_ROUND"
        return "OVER_SHELF_PACK_TOTAL"

df_clean["billunit_dq_flag"] = df_clean.apply(tag_bu_mismatch, axis=1)
print(df_clean["billunit_dq_flag"].value_counts().to_string())

# Step 5 - Derive concentration column for ranibizumab-type drugs
# concentration = (PKG SIZE * 1000) / (BILLUNITS * (HCPCS_dose_numeric))
# For most drugs this isn't cleanly derivable; set it for the ranibizumab group
def derive_conc(row):
    """mg/mL concentration for intravitreal injectables with 0.05 mL vials."""
    if row["HCPCS_dosage"] != "0.1 MG":
        return None
    if row["PKG_SIZE"] != 0.05:
        return None
    if row["BILLUNITS"] in (3.0, 5.0):
        return row["BILLUNITS"] * 0.1 / 0.05
    return None

df_clean["conc_mg_per_ml"] = df_clean.apply(derive_conc, axis=1)

# Step 6 - Add metadata
df_clean["source_file"]        = "october_2026_asp_ndc_hcpcs_crosswalk_091126"
df_clean["effective_start"]    = str(SOURCE_EFFECTIVE_START)
df_clean["effective_end"]      = str(SOURCE_EFFECTIVE_END)
df_clean["published_date"]     = str(SOURCE_PUBLISHED)

print(f"\nCleaned dataset: {len(df_clean):,} rows, {len(df_clean.columns)} columns")

After NDC format exclusion: 7,402 rows  (excluded: 1)
billunit_dq_flag
OK                       7194
OVER_CEILING_ROUND        139
OVER_SHELF_PACK_TOTAL      49
UNDER_CMS_DIRECT           20

Cleaned dataset: 7,402 rows, 17 columns


---
## 5 - Build the target-drug filtered table

This is the table the duplicate detection notebooks will actually query. I did a semantic enrichment including molecule name, brand family, drug type, and
therapeutic area.

In [0]:
TARGET_META = {
    # code: (molecule, brand_family, drug_type, delivery_route, therapeutic_area, reference_code)
    "J0178":  ("aflibercept",    "Eylea",         "originator",      "intravitreal", "ophthalmology",  None),
    "J1745":  ("infliximab",     "Remicade",       "originator",      "IV",           "autoimmune",     None),
    "J2350":  ("ocrelizumab",    "Ocrevus",        "originator",      "IV",           "neurology",      None),
    "J2351":  ("ocrelizumab",    "Ocrevus Zunovo", "new_formulation", "SC",           "neurology",      "J2350"),
    "J2778":  ("ranibizumab",    "Lucentis",       "originator",      "intravitreal", "ophthalmology",  None),
    "J9035":  ("bevacizumab",    "Avastin",        "originator",      "IV",           "oncology",       None),
    "J9271":  ("pembrolizumab",  "Keytruda",       "originator",      "IV",           "oncology",       None),
    "J9277":  ("pembrolizumab",  "Keytruda Qlex",  "new_formulation", "SC",           "oncology",       "J9271"),
    "Q5103":  ("infliximab",     "Inflectra",      "biosimilar",      "IV",           "autoimmune",     "J1745"),
    "Q5104":  ("infliximab",     "Renflexis",      "biosimilar",      "IV",           "autoimmune",     "J1745"),
    "Q5107":  ("bevacizumab",    "Mvasi",          "biosimilar",      "IV",           "oncology",       "J9035"),
    "Q5118":  ("bevacizumab",    "Zirabev",        "biosimilar",      "IV",           "oncology",       "J9035"),
    "Q5121":  ("infliximab",     "Avsola",         "biosimilar",      "IV",           "autoimmune",     "J1745"),
    "Q5124":  ("ranibizumab",    "Byooviz",        "biosimilar",      "intravitreal", "ophthalmology",  "J2778"),
    "Q5128":  ("ranibizumab",    "Cimerli",        "biosimilar",      "intravitreal", "ophthalmology",  "J2778"),
    "Q5138":  ("aflibercept",    "Wezlana",        "biosimilar",      "IV",           "ophthalmology",  "J0178"),
}

df_target = df_clean[df_clean["HCPCS_CODE"].isin(TARGET_META)].copy()

# Add the semantic columns
meta_df = pd.DataFrame.from_dict(
    TARGET_META, orient="index",
    columns=["molecule", "brand_family", "drug_type", "delivery_route",
             "therapeutic_area", "reference_hcpcs"]
).reset_index().rename(columns={"index": "HCPCS_CODE"})

df_target = df_target.merge(meta_df, on="HCPCS_CODE", how="left")

print(f"Target rows: {len(df_target)}")
print()
print("By molecule and drug type:")
print(
    df_target.groupby(["molecule", "drug_type"])
    .agg(ndcs=("NDC", "count"), hcpcs_codes=("HCPCS_CODE", lambda x: ", ".join(sorted(x.unique()))))
    .reset_index()
    .sort_values(["molecule", "drug_type"])
    .to_string(index=False)
)

Target rows: 31

By molecule and drug type:
     molecule       drug_type  ndcs         hcpcs_codes
  aflibercept      biosimilar     1               Q5138
  aflibercept      originator     2               J0178
  bevacizumab      biosimilar     6        Q5107, Q5118
  bevacizumab      originator     2               J9035
   infliximab      biosimilar     4 Q5103, Q5104, Q5121
   infliximab      originator     2               J1745
  ocrelizumab new_formulation     1               J2351
  ocrelizumab      originator     1               J2350
pembrolizumab new_formulation     2               J9277
pembrolizumab      originator     2               J9271
  ranibizumab      biosimilar     6        Q5124, Q5128
  ranibizumab      originator     2               J2778


In [0]:
# Preview target table
display(df_target[[
    "HCPCS_CODE", "molecule", "brand_family", "drug_type",
    "delivery_route", "therapeutic_area",
    "NDC", "Drug_Name", "LABELER_NAME",
    "HCPCS_dosage", "PKG_SIZE", "BILLUNITS", "BILLUNITSPKG",
    "conc_mg_per_ml", "reference_hcpcs"
]].sort_values(["molecule", "HCPCS_CODE", "NDC"]))

HCPCS_CODE,molecule,brand_family,drug_type,delivery_route,therapeutic_area,NDC,Drug_Name,LABELER_NAME,HCPCS_dosage,PKG_SIZE,BILLUNITS,BILLUNITSPKG,conc_mg_per_ml,reference_hcpcs
J0178,aflibercept,Eylea,originator,intravitreal,ophthalmology,61755-0005-01,Eylea,"Regeneron Pharmaceuticals, Inc.",1 MG,0.05,2.0,2.0,null,null
J0178,aflibercept,Eylea,originator,intravitreal,ophthalmology,61755-0005-02,Eylea,"Regeneron Pharmaceuticals, Inc.",1 MG,0.05,2.0,2.0,null,null
Q5138,aflibercept,Wezlana,biosimilar,IV,ophthalmology,84612-0066-01,Wezlana,Optum Health Solutions Ireland Limited,1 MG,26.0,130.0,130.0,null,J0178
J9035,bevacizumab,Avastin,originator,IV,oncology,50242-0060-01,Avastin,"Genentech, Inc.",10 MG,4.0,10.0,10.0,null,null
J9035,bevacizumab,Avastin,originator,IV,oncology,50242-0061-01,Avastin,"Genentech, Inc.",10 MG,16.0,40.0,40.0,null,null
Q5107,bevacizumab,Mvasi,biosimilar,IV,oncology,55513-0206-01,Mvasi,Amgen Inc,10 MG,4.0,10.0,10.0,null,J9035
Q5107,bevacizumab,Mvasi,biosimilar,IV,oncology,55513-0206-21,Mvasi,Amgen Inc,10 MG,4.0,10.0,10.0,null,J9035
Q5107,bevacizumab,Mvasi,biosimilar,IV,oncology,55513-0207-01,Mvasi,Amgen Inc,10 MG,16.0,40.0,40.0,null,J9035
Q5107,bevacizumab,Mvasi,biosimilar,IV,oncology,55513-0207-21,Mvasi,Amgen Inc,10 MG,16.0,40.0,40.0,null,J9035
Q5118,bevacizumab,Zirabev,biosimilar,IV,oncology,00069-0315-01,Zirabev,Pfizer Inc,10 MG,4.0,10.0,10.0,null,J9035


---
## 6 - Write to Delta Lake - finally

In [0]:
# Create schema
spark.sql("CREATE SCHEMA IF NOT EXISTS cross_benefit_audit")
print("Schema `cross_benefit_audit` ready.")

Schema `cross_benefit_audit` ready.


In [0]:
# 6a - Full crosswalk table

FULL_TABLE = "cross_benefit_audit.cms_ndc_hcpcs_crosswalk"

sdf_full = spark.createDataFrame(df_clean)

(
    sdf_full
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(FULL_TABLE)
)

print(f"Written: {FULL_TABLE}")
print(f"Row count: {spark.table(FULL_TABLE).count():,}")

Written: cross_benefit_audit.cms_ndc_hcpcs_crosswalk
Row count: 7,402


In [0]:
# 6b - Target HCPCS crosswalk (project scope)

TARGET_TABLE = "cross_benefit_audit.target_hcpcs_crosswalk"

sdf_target = spark.createDataFrame(df_target)

(
    sdf_target
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET_TABLE)
)

print(f"Written: {TARGET_TABLE}")
print(f"Row count: {spark.table(TARGET_TABLE).count():,}")

Written: cross_benefit_audit.target_hcpcs_crosswalk
Row count: 31


In [0]:
# 6c - Describe final schemas

print(f"=== {FULL_TABLE} schema ===")
spark.table(FULL_TABLE).printSchema()

print(f"\n=== {TARGET_TABLE} schema ===")
spark.table(TARGET_TABLE).printSchema()

=== cross_benefit_audit.cms_ndc_hcpcs_crosswalk schema ===
root
 |-- HCPCS_CODE: string (nullable = true)
 |-- Short_Description: string (nullable = true)
 |-- LABELER_NAME: string (nullable = true)
 |-- NDC: string (nullable = true)
 |-- Drug_Name: string (nullable = true)
 |-- HCPCS_dosage: string (nullable = true)
 |-- PKG_SIZE: double (nullable = true)
 |-- PKG_QTY: long (nullable = true)
 |-- BILLUNITS: double (nullable = true)
 |-- BILLUNITSPKG: double (nullable = true)
 |-- ndc_valid: boolean (nullable = true)
 |-- billunit_dq_flag: string (nullable = true)
 |-- conc_mg_per_ml: double (nullable = true)
 |-- source_file: string (nullable = true)
 |-- effective_start: string (nullable = true)
 |-- effective_end: string (nullable = true)
 |-- published_date: string (nullable = true)


=== cross_benefit_audit.target_hcpcs_crosswalk schema ===
root
 |-- HCPCS_CODE: string (nullable = true)
 |-- Short_Description: string (nullable = true)
 |-- LABELER_NAME: string (nullable = true)
 |

---
## 7 - Sanity checks on the newly written tables

In [0]:
# Row counts
print("Row counts:")
print(f"  {FULL_TABLE:<50}: {spark.table(FULL_TABLE).count():>6,}")
print(f"  {TARGET_TABLE:<50}: {spark.table(TARGET_TABLE).count():>6,}")

Row counts:
  cross_benefit_audit.cms_ndc_hcpcs_crosswalk       :  7,402
  cross_benefit_audit.target_hcpcs_crosswalk        :     31


In [0]:
# Verify that all 16 target HCPCS codes are present
target_codes_in_table = (
    spark.table(TARGET_TABLE)
    .select("HCPCS_CODE")
    .distinct()
    .orderBy("HCPCS_CODE")
    .collect()
)
print(f"HCPCS codes in target table ({len(target_codes_in_table)}):")
for row in target_codes_in_table:
    print(f"  {row.HCPCS_CODE}")

HCPCS codes in target table (16):
  J0178
  J1745
  J2350
  J2351
  J2778
  J9035
  J9271
  J9277
  Q5103
  Q5104
  Q5107
  Q5118
  Q5121
  Q5124
  Q5128
  Q5138


In [0]:
# Verify no null NDCs in target table
null_ndcs = spark.table(TARGET_TABLE).filter(F.col("NDC").isNull()).count()
print(f"Null NDCs in target table: {null_ndcs}")  # should be 0

Null NDCs in target table: 0


In [0]:
# Verify the bad NDC was excluded from full table
bad_ndc_check = (
    spark.table(FULL_TABLE)
    .filter(F.col("NDC") == "888867413689")
    .count()
)
print(f"Non-standard NDC rows in full table: {bad_ndc_check}")  # should be 0

Non-standard NDC rows in full table: 0


In [0]:
# Keytruda (J9271) check
print("Keytruda rows in target table:")
(
    spark.table(TARGET_TABLE)
    .filter(F.col("HCPCS_CODE").isin("J9271", "J9277"))
    .select("HCPCS_CODE", "molecule", "brand_family", "drug_type",
            "delivery_route", "NDC", "BILLUNITS", "BILLUNITSPKG")
    .orderBy("HCPCS_CODE", "NDC")
    .show(truncate=False)
)

Keytruda rows in target table:
+----------+-------------+-------------+---------------+--------------+-------------+---------+------------+
|HCPCS_CODE|molecule     |brand_family |drug_type      |delivery_route|NDC          |BILLUNITS|BILLUNITSPKG|
+----------+-------------+-------------+---------------+--------------+-------------+---------+------------+
|J9271     |pembrolizumab|Keytruda     |originator     |IV            |00006-3026-02|100.0    |100.0       |
|J9271     |pembrolizumab|Keytruda     |originator     |IV            |00006-3026-04|100.0    |200.0       |
|J9277     |pembrolizumab|Keytruda Qlex|new_formulation|SC            |00006-3083-01|395.0    |395.0       |
|J9277     |pembrolizumab|Keytruda Qlex|new_formulation|SC            |00006-5083-01|790.0    |790.0       |
+----------+-------------+-------------+---------------+--------------+-------------+---------+------------+



In [0]:
# Ranibizumab billing unit split check
print("Ranibizumab billing unit split:")
(
    spark.table(TARGET_TABLE)
    .filter(F.col("molecule") == "ranibizumab")
    .select("HCPCS_CODE", "Drug_Name", "NDC",
            "BILLUNITS", "BILLUNITSPKG", "conc_mg_per_ml", "drug_type")
    .orderBy("HCPCS_CODE", "NDC")
    .show(truncate=False)
)

Ranibizumab billing unit split:
+----------+------------+-------------+---------+------------+-----------------+----------+
|HCPCS_CODE|Drug_Name   |NDC          |BILLUNITS|BILLUNITSPKG|conc_mg_per_ml   |drug_type |
+----------+------------+-------------+---------+------------+-----------------+----------+
|J2778     |Lucentis    |50242-0080-03|5.0      |5.0         |10.0             |originator|
|J2778     |Lucentis PFS|50242-0082-03|3.0      |3.0         |6.000000000000001|originator|
|Q5124     |Byooviz     |64406-0019-01|5.0      |5.0         |10.0             |biosimilar|
|Q5124     |Byooviz     |82667-0018-05|5.0      |5.0         |10.0             |biosimilar|
|Q5128     |Cimerli     |61314-0624-94|3.0      |3.0         |6.000000000000001|biosimilar|
|Q5128     |Cimerli     |61314-0625-94|5.0      |5.0         |10.0             |biosimilar|
|Q5128     |Cimerli     |70114-0440-01|3.0      |3.0         |6.000000000000001|biosimilar|
|Q5128     |Cimerli     |70114-0441-01|5.0      

---
## 8 - Summary

### What was built

| Table | Rows | Description |
|---|---|---|
| `cross_benefit_audit.cms_ndc_hcpcs_crosswalk` | 7,402 | Full cleaned October 2026 ASP crosswalk. One row excluded (SynoJoynt malformed NDC). |
| `cross_benefit_audit.target_hcpcs_crosswalk` | 31 | Project-scoped crosswalk for 6 specialty drugs: 6 originator codes, 8 biosimilar Q-codes, 2 new-formulation codes. |

### Key DQ findings documented

1. **One malformed NDC** (SynoJoynt GS1 barcode) - excluded
2. **Zero true duplicate rows** - all 319 multi-NDC rows are legitimate multi-HCPCS mappings
3. **208 arithmetic mismatches** in `BILLUNITS × PKG_QTY vs BILLUNITSPKG` - all explained by CMS methodology (ceiling rounding, oral unit-dose convention, shelf-pack totals). Our 6 target drugs are unaffected.
4. **Ranibizumab concentration split** (BILLUNITS=3 vs 5) - clinically valid; two distinct formulations (6 mg/mL and 10 mg/mL)
5. **Biosimilars under separate Q-codes** - detection logic must include Q5103/Q5104/Q5107/Q5118/Q5121/Q5124/Q5128/Q5138 alongside the originator J-codes

### What comes next

- **NB02** - Synthetic pharmacy claims: generate realistic NCPDP-style claim records for our 6 drugs, incorporating the crosswalk NDCs, realistic dispense dates, and both originator and biosimilar dispensing patterns
- **NB03** - Synthetic medical claims: generate CMS-1500/UB-04-style infusion claims keyed to the same HCPCS codes, with administration dates that lag or overlap pharmacy dispense dates by clinically realistic windows
- **NB04** - Matching engine: implement NDC→HCPCS bridge, dose-quantity reconciliation, and fuzzy date-window matching. Demonstrate the Q-code gap as a failure mode of naive matching.
- **NB05** - Risk stratification: NPI co-location analysis, specialty pharmacy flag, high-cost molecule flag, biosimilar substitution pattern detection
- **NB06** - Reporting: per-member, per-drug, per-payer summary tables for payment integrity review

---
*Sources:*
*CMS ASP NDC-HCPCS Crosswalk - https://www.cms.gov/medicare/physician-fee-schedule/asp-drug-pricing*
*CMS Part B Drug Pricing File, October 2026 Quarter*